# Build-2 Assist retrieval — runs against the **Build-1 Lakebase Search index**

This notebook **executes** the exact retrieval the app's `search_playbooks` tool performs, proving Assist memo drafting pulls from the Build-1 **Lakebase Search index** (`public.reference_playbooks`, a Postgres `tsvector` full-text index) — **not a separate vector store**.

Connection = the same Lakebase branch the deployed app uses (`projects/sentinel-payments/branches/production`), via a Lakebase OAuth token.

In [1]:
import os, json, subprocess, psycopg, pandas as pd
PGHOST = 'ep-wandering-boat-d83925kg.database.us-east-2.cloud.databricks.com'
PGUSER = 'scott.johnson@databricks.com'
EP = 'projects/sentinel-payments/branches/production/endpoints/primary'
PROFILE = os.environ.get('DATABRICKS_CONFIG_PROFILE', 'fe-tech')
# Mint a short-lived Lakebase OAuth token for the Autoscale endpoint (same
# credential mechanism the deployed app's pool uses).
cred = json.loads(subprocess.check_output(
    ['databricks','postgres','generate-database-credential', EP, '--profile', PROFILE, '-o','json']))
conn = psycopg.connect(host=PGHOST, dbname='databricks_postgres', user=PGUSER, password=cred['token'], sslmode='require')
print('Connected to Lakebase branch: projects/sentinel-payments/branches/production')

Connected to Lakebase branch: projects/sentinel-payments/branches/production


## 1. The index is a Postgres `tsvector` living IN Lakebase (not an external vector store)

In [2]:
meta = pd.read_sql("""SELECT table_schema, table_name, column_name, data_type
  FROM information_schema.columns
  WHERE table_schema='public' AND table_name='reference_playbooks'
    AND column_name IN ('search_vector','embedding') ORDER BY column_name""", conn)
n = pd.read_sql('SELECT COUNT(*) AS playbooks FROM public.reference_playbooks', conn)
print('Index: public.reference_playbooks (Databricks Lakebase / Postgres) | rows:', int(n.playbooks[0]))
meta

Index: public.reference_playbooks (Databricks Lakebase / Postgres) | rows: 8


,table_schema,table_name,column_name,data_type
0,public,reference_playbooks,embedding,USER-DEFINED
1,public,reference_playbooks,search_vector,tsvector


## 2. Run the retrieval — full-text search over `search_vector` for the hero's signals
The exact query `searchPlaybooks()` / the `search_playbooks` agent tool runs (OR-token `to_tsquery`, `ts_rank`, `signal_type` pinned first).

In [3]:
signal_type = 'duplicate_identity'
tsquery = ' | '.join(['duplicate','identity','cross','agency','fraud','match','high','risk'])
fts = """SELECT playbook_id, signal_type, risk_level, title, regulatory_cite,
  ts_rank(search_vector, to_tsquery('english', %(tsq)s)) AS ts_rank
  FROM public.reference_playbooks
  WHERE search_vector @@ to_tsquery('english', %(tsq)s) OR signal_type = %(sig)s
  ORDER BY CASE WHEN signal_type = %(sig)s THEN 0 ELSE 1 END, ts_rank DESC LIMIT 3"""
hits = pd.read_sql(fts, conn, params={'tsq': tsquery, 'sig': signal_type})
print('Retrieved', len(hits), 'playbooks from the Lakebase Search index')
hits

Retrieved 3 playbooks from the Lakebase Search index


,playbook_id,signal_type,risk_level,title,regulatory_cite,ts_rank
0,2,duplicate_identity,high,Duplicate Identity Detection Protocol,"IPERA 2010, 42 USC §1320a-7a",0.037415
1,1,cross_agency_fraud_flag,high,Cross-Agency Fraud Match Response,"31 USC §3321, OMB Circular A-123 Appendix C",0.050872
2,7,residence_mismatch,low,Residence Verification Protocol,SNAP 7 CFR §273.2(f),0.017098


## 3. The retrieved governing policy (what the drafted memo cites)

In [4]:
steps = pd.read_sql("""SELECT signal_type, title, regulatory_cite, verification_steps
  FROM public.reference_playbooks WHERE playbook_id = ANY(%(ids)s)""",
  conn, params={'ids': hits.playbook_id.tolist()})
for _, r in steps.iterrows():
    print(f'[{r.signal_type}] {r.title} - cite: {r.regulatory_cite}')
print()
print('separate_vector_store_used = False  (retrieval is Lakebase-native Postgres FTS)')
conn.close()

[cross_agency_fraud_flag] Cross-Agency Fraud Match Response - cite: 31 USC §3321, OMB Circular A-123 Appendix C
[duplicate_identity] Duplicate Identity Detection Protocol - cite: IPERA 2010, 42 USC §1320a-7a
[residence_mismatch] Residence Verification Protocol - cite: SNAP 7 CFR §273.2(f)

separate_vector_store_used = False  (retrieval is Lakebase-native Postgres FTS)
